# CSCIE89 Problem 2: RNN with validation, lookback comparison, and five-day targets

This notebook starts from the Problem 1 RNN notebook for Micron Technology (MU) and makes three changes:

- **(a) Validation split.** The original training period (January 2016 to September 6, 2026) is split chronologically: the first 80% of rows train the model and the last 20% validate it. The scaler is fitted on the training portion only and reused, unchanged, for validation and test prices.
- **(b) Lookback comparison.** The same architecture is trained with lookback windows of 10, 30, 60, and 120 trading observations, each predicting the next opening price.
- **(c) Five-day targets.** Inputs keep the shape `(60, 1)`, and each target contains the **next five** opening prices. The output layer produces five values.

Every configuration runs through one function, `run_experiment(lookback, horizon)`, so experiments differ only in how the windows are built. To add a configuration, edit `LOOKBACKS` or `MULTI_STEP` in Step 1; the comparison tables and plots pick it up automatically.

## 1. Dependencies and configuration
Select a Python kernel with PyTorch, NumPy, pandas, matplotlib, and scikit-learn installed. Set `DATA_DIR` to the folder containing `MU.csv` and `MU_test.csv`.

Five configurations are trained for `EPOCHS` epochs each, so a full run takes roughly five times as long as the Problem 1 notebook (the 120-step lookback is the slowest). Set `EPOCHS = 5` for a quick end-to-end check before the full run.

In [ ]:
# %pip install torch numpy pandas matplotlib scikit-learn

In [ ]:
# Step 1: Import libraries, choose hyperparameters, and select a device.
from pathlib import Path
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

DATA_DIR = Path.cwd()/"data"  # Change this if your CSV files are in another directory.
OUTPUT_DIR = Path.cwd()/"output"
HIDDEN_SIZE = 50
DROPOUT = 0.2
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.001
MAX_GRAD_NORM = 1.0  # Clip the combined gradient norm before each Adam update.
SEED = 42

# Problem 2 settings.
VALIDATION_FRACTION = 0.2      # (a) The last 20% of the original training period.
LOOKBACKS = [10, 30, 60, 120]  # (b) One-day-ahead targets for each lookback.
MULTI_STEP = [(60, 5)]         # (c) (lookback, horizon): (60, 1) inputs, five future opens.
EXPERIMENTS = [(lookback, 1) for lookback in LOOKBACKS] + MULTI_STEP
RESTORE_BEST_WEIGHTS = True    # Keep the epoch with the lowest validation loss.
PRICE_TRANSFORM = "none"       # "none" follows the assignment; "log" scales log prices instead.

# Fixed colors per experiment, so each configuration keeps its color in every plot.
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SERIES_MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
ACTUAL_COLOR = "#0b0b0b"
BASELINE_COLOR = "#52514e"


def set_seed(seed):
    # Seeding makes runs more repeatable, but different devices/backends can still
    # produce different floating-point results; exact reproducibility is not promised.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

# CUDA is used on supported NVIDIA GPUs, MPS on supported Macs, otherwise CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"PyTorch {torch.__version__}; device: {device}")
print("Experiments (lookback, horizon):", EXPERIMENTS)

## 2. Load, clean, and split the time series chronologically
Loading and cleaning are unchanged from Problem 1. The original training file is then split **by position in time**, not at random: the earliest 80% of rows form the training portion and the latest 20% form the validation portion. All training dates precede all validation dates, which precede all test dates.

In [ ]:
# Step 2: Find CSV files, clean prices, and split the training period 80/20 in time.
def find_csv(*names):
    data_dir = Path(DATA_DIR).expanduser()
    for name in names:
        path = data_dir / name
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"Expected one of {names} in {data_dir.resolve()}. "
        "Copy the CSV files there or change DATA_DIR."
    )


def load_prices(path):
    # Keep only the two columns used by this forecasting exercise.
    frame = pd.read_csv(path)
    frame.columns = frame.columns.str.strip()
    if not {"Date", "Open"}.issubset(frame.columns):
        raise ValueError(f"{path.name} must contain Date and Open columns.")
    frame = frame[["Date", "Open"]].copy()
    frame["Date"] = frame["Date"].map(lambda value: pd.to_datetime(value, errors="coerce"))
    frame["Open"] = pd.to_numeric(
        frame["Open"].astype(str).str.replace(",", "", regex=False), errors="coerce"
    )
    # Remove non-price rows, then order observations from oldest to newest.
    frame = frame.dropna(subset=["Date", "Open"])
    frame = frame.sort_values("Date").reset_index(drop=True)
    if frame.empty or not np.isfinite(frame["Open"].to_numpy()).all():
        raise ValueError(f"{path.name} contains no usable prices or contains infinite values.")
    if frame["Date"].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate dates; resolve them before training.")
    return frame


mu_full_train = load_prices(find_csv("MU.csv"))
mu_test = load_prices(find_csv("MU_test.csv"))
if mu_full_train["Date"].max() >= mu_test["Date"].min():
    raise ValueError("All training dates must precede all test dates.")

# (a) Chronological split of the original training period.
split_index = int(len(mu_full_train) * (1 - VALIDATION_FRACTION))
mu_train = mu_full_train.iloc[:split_index].reset_index(drop=True)
mu_val = mu_full_train.iloc[split_index:].reset_index(drop=True)

max_lookback = max(lookback for lookback, _ in EXPERIMENTS)
max_horizon = max(horizon for _, horizon in EXPERIMENTS)
if len(mu_train) < max_lookback + max_horizon:
    raise ValueError(f"The training portion needs at least {max_lookback + max_horizon} rows.")
if min(len(mu_val), len(mu_test)) < max_horizon:
    raise ValueError(f"Validation and test periods need at least {max_horizon} rows.")

for name, frame in [("Training", mu_train), ("Validation", mu_val), ("Test", mu_test)]:
    print(f"{name:<10} rows: {len(frame):5d}   "
          f"{frame['Date'].min():%Y-%m-%d} to {frame['Date'].max():%Y-%m-%d}")

# Opening prices as (rows, 1) float32 arrays, the shape the scaler expects.
train_open = mu_train[["Open"]].to_numpy(dtype=np.float32)
val_open = mu_val[["Open"]].to_numpy(dtype=np.float32)
test_open = mu_test[["Open"]].to_numpy(dtype=np.float32)
full_train_open = np.concatenate([train_open, val_open])  # Context for the test windows.

In [ ]:
# Plot the three periods to see where the split falls relative to the price history.
plt.figure(figsize=(10, 4))
for (name, frame), color in zip(
    [("Training", mu_train), ("Validation", mu_val), ("Test", mu_test)], SERIES_COLORS
):
    plt.plot(frame["Date"], frame["Open"], color=color, linewidth=1.5, label=name)
plt.title("Micron (MU) opening prices by period")
plt.ylabel("Opening price (USD)")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Scale prices using the training portion only
The scaler is fitted on the **training portion only**, then applied unchanged to validation and test prices. Validation prices therefore play the same role as unseen future prices.

**Check the printed scaled ranges.** MU rose sharply in 2025 and 2026, so validation and test prices can lie far above the training maximum, giving scaled values well above 1. The model must then predict values it never saw during training, and tanh-based layers tend to flatten out at high inputs. If this happens, it is a result worth reporting, not a bug. Setting `PRICE_TRANSFORM = "log"` scales log prices instead, which compresses the range; the scaler is still fitted on the training portion only.

In [ ]:
# Step 3: Fit the scaler on the training portion; reuse it for validation and test.
if PRICE_TRANSFORM not in ("none", "log"):
    raise ValueError('PRICE_TRANSFORM must be "none" or "log".')


def forward_transform(prices):
    return np.log(prices) if PRICE_TRANSFORM == "log" else prices


def inverse_transform(values):
    return np.exp(values) if PRICE_TRANSFORM == "log" else values


scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(forward_transform(train_open))  # (a) Fitted on the training portion only.


def to_scaled(prices):
    return scaler.transform(forward_transform(prices)).astype(np.float32)


def to_dollars(scaled):
    # Works for any shape, such as (n, 1) or (n, horizon) predictions.
    scaled = np.asarray(scaled)
    values = scaler.inverse_transform(scaled.reshape(-1, 1))
    return inverse_transform(values).reshape(scaled.shape)


train_scaled = to_scaled(train_open)
val_scaled = to_scaled(val_open)
test_scaled = to_scaled(test_open)
full_train_scaled = np.concatenate([train_scaled, val_scaled])

for name, values in [("Training", train_scaled), ("Validation", val_scaled), ("Test", test_scaled)]:
    print(f"{name:<10} scaled range: {values.min():7.3f} to {values.max():7.3f}")
if max(val_scaled.max(), test_scaled.max()) > 1:
    print("Note: validation or test prices exceed the training maximum (scaled values above 1); "
          "the model must extrapolate beyond its training range.")

## 4. Build windows for any lookback and horizon
Two functions replace the hard-coded window construction from Problem 1:

- `make_windows(scaled, lookback, horizon)` builds training windows from one contiguous series. For a target starting at position `i`, the input is positions `i - lookback` through `i - 1`, with shape `(lookback, 1)`, and the target is positions `i` through `i + horizon - 1`, with shape `(horizon,)`. A series of `N` rows gives `N - lookback - horizon + 1` windows.
- `make_eval_windows(history, eval, lookback, horizon)` builds **rolling** evaluation windows, as in Problem 1's test step. The first window ends with the last `lookback` values before the evaluation period; later windows include actual evaluation prices observed before their targets. Only windows whose full target lies inside the evaluation period are kept, so a five-day horizon gives four fewer windows than a one-day horizon.

Validation windows draw their context from the training portion; test windows draw theirs from the full original training period, whose last rows are the validation portion. For a given horizon, every lookback is therefore evaluated on exactly the same target days.

The persistence baseline repeats the last observed opening price for every day of the horizon. It is computed for each experiment on the same windows, so the ratio `MAE / baseline MAE` is comparable across experiments: values below 1 mean the model beats the baseline.

In [ ]:
# Step 4: Window construction for any (lookback, horizon), plus the persistence baseline.
def make_windows(scaled, lookback, horizon):
    # Inputs: (n, lookback, 1). Targets: (n, horizon), the values right after each input.
    starts = range(lookback, len(scaled) - horizon + 1)
    X = np.stack([scaled[i - lookback:i] for i in starts])
    y = np.stack([scaled[i:i + horizon, 0] for i in starts])
    return X, y


def make_eval_windows(history_scaled, eval_scaled, lookback, horizon):
    # Prepend the last `lookback` history values so the first target has a full window.
    context = np.concatenate([history_scaled[-lookback:], eval_scaled])
    return make_windows(context, lookback, horizon)


def eval_targets(eval_prices, horizon):
    # Actual future opens in dollars for each evaluation window: (n, horizon).
    starts = range(len(eval_prices) - horizon + 1)
    return np.stack([eval_prices[i:i + horizon, 0] for i in starts])


def persistence_baseline(history_prices, eval_prices, horizon):
    # The last open observed before each window's first target, repeated across the horizon.
    previous = np.concatenate([history_prices[-1:], eval_prices[:-1]])[:, 0]
    n_windows = len(eval_prices) - horizon + 1
    return np.repeat(previous[:n_windows, None], horizon, axis=1)


# Shape check for every configuration before any training.
for lookback, horizon in EXPERIMENTS:
    X_tr, y_tr = make_windows(train_scaled, lookback, horizon)
    X_va, y_va = make_eval_windows(train_scaled, val_scaled, lookback, horizon)
    X_te, y_te = make_eval_windows(full_train_scaled, test_scaled, lookback, horizon)
    # Evaluation targets must line up with the actual prices they represent.
    np.testing.assert_allclose(to_dollars(y_va), eval_targets(val_open, horizon), rtol=1e-4)
    np.testing.assert_allclose(to_dollars(y_te), eval_targets(test_open, horizon), rtol=1e-4)
    print(f"lookback {lookback:3d}, horizon {horizon}: "
          f"train X {X_tr.shape}, y {y_tr.shape} | "
          f"validation X {X_va.shape} | test X {X_te.shape}")

## 5. Define the four-layer vanilla RNN
The architecture is unchanged from Problem 1 except for one argument: `output_size`, the number of values the final linear layer produces. It equals the horizon, so one-day experiments output `(B, 1)` and the five-day experiment outputs `(B, 5)`. The lookback is not a model argument: each RNN layer runs one step per time step in its input, so the same class handles windows of 10 to 120 steps.

| Operation | Shape |
|---|---|
| Input batch (`batch_first=True`) | `(B, lookback, 1)` |
| Each RNN's sequence output | `(B, lookback, 50)` |
| Fourth layer's last sequence output | `(B, 50)` |
| Linear regression output | `(B, horizon)` |

In [ ]:
class MuRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, dropout=0.2, output_size=1):
        # Initialize nn.Module so PyTorch tracks child layers and parameters.
        super().__init__()

        # Each module processes a full sequence using a tanh recurrence.
        # batch_first=True means inputs/outputs use (batch, time, features).
        self.rnn1 = nn.RNN(input_size, hidden_size, num_layers=1,
                           nonlinearity="tanh", batch_first=True)
        self.rnn2 = nn.RNN(hidden_size, hidden_size, num_layers=1,
                           nonlinearity="tanh", batch_first=True)
        self.rnn3 = nn.RNN(hidden_size, hidden_size, num_layers=1,
                           nonlinearity="tanh", batch_first=True)
        self.rnn4 = nn.RNN(hidden_size, hidden_size, num_layers=1,
                           nonlinearity="tanh", batch_first=True)

        # Explicit dropout randomly zeros 20% of activations during training.
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        self.dropout4 = nn.Dropout(dropout)
        # One output per forecast day: 1 for next-day targets, 5 for five-day targets.
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Input x: (B, lookback, 1). No h_0 is supplied, so each RNN starts
        # with zeros for every example.
        x, _ = self.rnn1(x)      # Sequence: (B, lookback, hidden_size).
        x = self.dropout1(x)
        x, _ = self.rnn2(x)
        x = self.dropout2(x)
        x, _ = self.rnn3(x)
        x = self.dropout3(x)
        x, _ = self.rnn4(x)

        # Select the last TIME step for every example: (B, lookback, 50) -> (B, 50).
        x = x[:, -1, :]
        x = self.dropout4(x)
        # (B, 50) -> (B, output_size). No sigmoid: scaled prices can exceed 1.
        return self.output(x)


# Parameter count for each output size used in the experiments.
for horizon in sorted({horizon for _, horizon in EXPERIMENTS}):
    candidate = MuRNN(hidden_size=HIDDEN_SIZE, dropout=DROPOUT, output_size=horizon)
    print(f"horizon {horizon}: {sum(p.numel() for p in candidate.parameters()):,} trainable parameters")

## 6. Training and evaluation functions
`train_model` is the Problem 1 training loop with one addition: after every epoch it computes the **validation MSE** on scaled prices with dropout disabled. With `RESTORE_BEST_WEIGHTS = True`, the weights from the epoch with the lowest validation loss are restored after training, which is the main practical use of a validation set. Set it to `False` to keep the final-epoch weights, as in Problem 1.

`run_experiment(lookback, horizon)` builds the windows, trains a freshly seeded model, evaluates it on the validation and test periods in dollars, and saves a checkpoint. Each experiment is reseeded, so differences between experiments come from the data construction, not from different random initializations or batch orders.

In [ ]:
# Step 6: Reusable training, prediction, evaluation, and checkpoint functions.
def make_loader(X, y):
    # A dedicated random generator makes the order of shuffled batches repeatable.
    dataset = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).float())
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                      generator=generator, num_workers=0)


def predict_scaled(model, X):
    # Chronological predictions with dropout and gradient tracking disabled.
    model.eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(X).float()),
                        batch_size=BATCH_SIZE, shuffle=False)
    batches = []
    with torch.inference_mode():
        for (batch_x,) in loader:
            batches.append(model(batch_x.to(device)).cpu().numpy())
    return np.concatenate(batches, axis=0)


def train_model(model, train_loader, X_val, y_val):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history = {"loss": [], "val_loss": []}
    best = {"val_loss": float("inf"), "epoch": 0, "state": None}

    for epoch in range(EPOCHS):
        model.train()  # Enables dropout.
        total_squared_error = 0.0
        examples_seen = 0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            batch_predictions = model(batch_x)
            assert batch_predictions.shape == batch_y.shape  # (B, horizon) on both sides.
            loss = criterion(batch_predictions, batch_y)
            loss.backward()
            if MAX_GRAD_NORM is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            batch_count = batch_x.size(0)
            total_squared_error += loss.item() * batch_count
            examples_seen += batch_count

        history["loss"].append(total_squared_error / examples_seen)
        # Validation MSE on scaled prices, averaged over every window and forecast day.
        val_loss = float(np.mean((predict_scaled(model, X_val) - y_val) ** 2))
        history["val_loss"].append(val_loss)
        if val_loss < best["val_loss"]:
            best = {
                "val_loss": val_loss,
                "epoch": epoch + 1,
                "state": {key: value.detach().cpu().clone() for key, value in model.state_dict().items()},
            }
        if epoch == 0 or (epoch + 1) % 25 == 0 or epoch + 1 == EPOCHS:
            print(f"  Epoch {epoch + 1:3d}/{EPOCHS}: training MSE = {history['loss'][-1]:.6f}; "
                  f"validation MSE = {val_loss:.6f}")

    if RESTORE_BEST_WEIGHTS:
        model.load_state_dict(best["state"])
    return history, best["epoch"]


def error_metrics(predicted, actual):
    # Dollar errors for (n, horizon) arrays, overall and for each forecast day.
    errors = predicted - actual
    return {
        "mae": float(np.abs(errors).mean()),
        "rmse": float(np.sqrt(np.mean(errors ** 2))),
        "mae_by_day": np.abs(errors).mean(axis=0),
    }


def scaler_state(fitted):
    # Lists and primitive values can be loaded with weights_only=True.
    return {
        "feature_range": list(fitted.feature_range),
        "scale_": fitted.scale_.tolist(),
        "min_": fitted.min_.tolist(),
        "data_min_": fitted.data_min_.tolist(),
        "data_max_": fitted.data_max_.tolist(),
        "data_range_": fitted.data_range_.tolist(),
        "n_features_in_": int(fitted.n_features_in_),
        "n_samples_seen_": int(fitted.n_samples_seen_),
    }


def run_experiment(lookback, horizon):
    name = f"L{lookback}_H{horizon}"
    set_seed(SEED)  # Same initialization and batch order for every experiment.
    X_train, y_train = make_windows(train_scaled, lookback, horizon)
    X_val, y_val = make_eval_windows(train_scaled, val_scaled, lookback, horizon)
    X_test, _ = make_eval_windows(full_train_scaled, test_scaled, lookback, horizon)

    model_config = {"input_size": 1, "hidden_size": HIDDEN_SIZE,
                    "dropout": DROPOUT, "output_size": horizon}
    model = MuRNN(**model_config).to(device)
    start = time.perf_counter()
    history, best_epoch = train_model(model, make_loader(X_train, y_train), X_val, y_val)
    minutes = (time.perf_counter() - start) / 60

    result = {
        "name": name, "lookback": lookback, "horizon": horizon,
        "train_windows": len(X_train), "best_epoch": best_epoch, "minutes": minutes,
        "history": history, "model_config": model_config,
    }
    for split, X, history_open, eval_open, dates in [
        ("val", X_val, train_open, val_open, mu_val["Date"]),
        ("test", X_test, full_train_open, test_open, mu_test["Date"]),
    ]:
        actual = eval_targets(eval_open, horizon)
        predicted = to_dollars(predict_scaled(model, X))
        baseline = persistence_baseline(history_open, eval_open, horizon)
        assert predicted.shape == actual.shape == baseline.shape
        result[split] = {
            "dates": dates.iloc[:len(actual)].to_numpy(),  # Date of each window's first target.
            "actual": actual, "predicted": predicted, "baseline": baseline,
            "model": error_metrics(predicted, actual),
            "baseline_metrics": error_metrics(baseline, actual),
        }

    checkpoint_path = Path(OUTPUT_DIR).expanduser() / f"mu_rnn_{name}.pt"
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model_state_dict": {key: value.detach().cpu().clone() for key, value in model.state_dict().items()},
        "model_config": model_config,
        "lookback": lookback,
        "horizon": horizon,
        "best_epoch": best_epoch,
        "price_transform": PRICE_TRANSFORM,
        "history": history,
        "scaler": scaler_state(scaler),
    }, checkpoint_path)
    result["checkpoint_path"] = checkpoint_path
    print(f"  Best epoch {best_epoch}; {minutes:.1f} min; "
          f"validation MAE ${result['val']['model']['mae']:.2f} "
          f"(baseline ${result['val']['baseline_metrics']['mae']:.2f})")
    return result, model

## 7. Run every experiment
Each configuration in `EXPERIMENTS` is trained and evaluated in turn. Results are stored in `results` (metrics, predictions, and loss histories) and `models` (the trained networks), keyed by names such as `L60_H1`.

In [ ]:
# Step 7: Train and evaluate each (lookback, horizon) configuration.
results, models = {}, {}
for lookback, horizon in EXPERIMENTS:
    print(f"=== lookback {lookback}, horizon {horizon} ===")
    result, trained_model = run_experiment(lookback, horizon)
    results[result["name"]] = result
    models[result["name"]] = trained_model

# Fixed color and marker per experiment, assigned in EXPERIMENTS order.
styles = {
    name: {"color": SERIES_COLORS[i % len(SERIES_COLORS)], "marker": SERIES_MARKERS[i % len(SERIES_MARKERS)]}
    for i, name in enumerate(results)
}

## 8. Compare experiments
### 8.1 Summary table
Errors are in dollars. `MAE / baseline` below 1 means the model beats the persistence baseline on the same windows.

**Choose a configuration using the validation columns only.** The test period has about 23 trading days (four fewer windows for a five-day horizon), so test metrics are noisy and serve as a final check rather than a basis for selection. For five-day experiments, MAE and RMSE average over all five forecast days; Section 8.4 breaks them out by day.

In [ ]:
# Step 8.1: One row per experiment.
rows = []
for r in results.values():
    row = {"experiment": r["name"], "lookback": r["lookback"], "horizon": r["horizon"],
           "train windows": r["train_windows"], "best epoch": r["best_epoch"],
           "minutes": r["minutes"]}
    for split, label in [("val", "validation"), ("test", "test")]:
        model_metrics, baseline_metrics = r[split]["model"], r[split]["baseline_metrics"]
        row[f"{label} MAE"] = model_metrics["mae"]
        row[f"{label} RMSE"] = model_metrics["rmse"]
        row[f"{label} baseline MAE"] = baseline_metrics["mae"]
        row[f"{label} MAE / baseline"] = model_metrics["mae"] / baseline_metrics["mae"]
    rows.append(row)

summary = pd.DataFrame(rows).set_index("experiment")
# Part (b) selection: among one-day experiments, all scored on the same validation days.
candidates = summary[summary["horizon"] == 1] if (summary["horizon"] == 1).any() else summary
best_name = candidates["validation MAE"].idxmin()
print(f"Lowest validation MAE among {'one-day' if candidates is not summary else 'all'} experiments: {best_name}")
summary.round(3)

### 8.2 Training and validation loss curves
Both losses are MSE on scaled prices, on a log scale. A validation curve that rises while the training curve keeps falling indicates overfitting; the circle on each validation curve marks the epoch whose weights were kept.

In [ ]:
# Step 8.2: Loss curves, one line per experiment, in two panels with the same scale type.
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for name, r in results.items():
    epochs = range(1, len(r["history"]["loss"]) + 1)
    axes[0].plot(epochs, r["history"]["loss"], color=styles[name]["color"], linewidth=1.5, label=name)
    axes[1].plot(epochs, r["history"]["val_loss"], color=styles[name]["color"], linewidth=1.5, label=name)
    if RESTORE_BEST_WEIGHTS:
        best = r["best_epoch"]
        axes[1].plot(best, r["history"]["val_loss"][best - 1], marker="o", markersize=8,
                     color=styles[name]["color"], markeredgecolor="white")
for ax, title in zip(axes, ["Training loss", "Validation loss"]):
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE (scaled prices)")
    ax.set_title(title)
    ax.grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

### 8.3 Effect of the lookback window (one-day targets)
Validation and test MAE for each lookback, with the persistence baseline as a dashed line. All lookbacks are scored on the same target days, so the baseline is identical across them.

In [ ]:
# Step 8.3: MAE against lookback for the one-day experiments.
one_day = summary[summary["horizon"] == 1].sort_values("lookback")
if one_day.empty:
    print("No one-day experiments to compare.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, label in zip(axes, ["validation", "test"]):
        ax.plot(one_day["lookback"], one_day[f"{label} MAE"], color=SERIES_COLORS[0],
                marker="o", markersize=8, linewidth=2, label="RNN")
        ax.axhline(one_day[f"{label} baseline MAE"].iloc[0], color=BASELINE_COLOR,
                   linestyle="--", linewidth=1.5, label="Previous-open baseline")
        for lookback, mae in zip(one_day["lookback"], one_day[f"{label} MAE"]):
            ax.annotate(f"${mae:,.2f}", (lookback, mae), textcoords="offset points",
                        xytext=(0, 8), ha="center", fontsize=9)
        ax.set_xscale("log")
        ax.set_xticks(one_day["lookback"])
        ax.set_xticklabels(one_day["lookback"])
        ax.set_xlabel("Lookback (trading observations)")
        ax.set_ylabel("MAE (USD)")
        ax.set_title(f"{label.capitalize()} MAE by lookback")
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

Rolling one-step-ahead predictions for each one-day experiment, one panel per lookback, over the validation and test periods. The validation panels show how each model behaves across the price range above the training maximum.

In [ ]:
# Step 8.3 (continued): Predictions against actual prices, one panel per one-day experiment.
def plot_one_day_predictions(split, title):
    names = [name for name, r in results.items() if r["horizon"] == 1]
    if not names:
        return
    columns = 2
    rows_needed = (len(names) + columns - 1) // columns
    fig, axes = plt.subplots(rows_needed, columns, figsize=(13, 3.6 * rows_needed),
                             sharex=True, sharey=True, squeeze=False)
    for ax, name in zip(axes.flat, names):
        data = results[name][split]
        ax.plot(data["dates"], data["actual"][:, 0], color=ACTUAL_COLOR, linewidth=1.5, label="Actual")
        ax.plot(data["dates"], data["baseline"][:, 0], color=BASELINE_COLOR, linestyle="--",
                linewidth=1.5, label="Previous-open baseline")
        ax.plot(data["dates"], data["predicted"][:, 0], color=styles[name]["color"],
                linewidth=2, label=f"RNN {name}")
        # Plain "USD" avoids matplotlib reading paired dollar signs as math markup.
        ax.set_title(f"{name}: MAE {data['model']['mae']:,.2f} USD "
                     f"(baseline {data['baseline_metrics']['mae']:,.2f} USD)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="upper left")
        ax.tick_params(axis="x", rotation=30)
    for ax in list(axes.flat)[len(names):]:
        ax.set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("Opening price (USD)")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_one_day_predictions("val", "Micron (MU) validation period: rolling one-step-ahead predictions")
plot_one_day_predictions("test", "Micron (MU) test period: rolling one-step-ahead predictions")

### 8.4 Five-day targets
For each multi-step experiment, the table and chart break the error out by forecast day: day 1 is the next trading day after the window, day 5 is four trading days later. Errors normally grow with the forecast day for both the model and the baseline.

To compare the five-day model's day 1 with the one-day model at the same lookback (for example `L60_H5` day 1 against `L60_H1`), note that the five-day model is scored on four fewer windows, so the comparison covers slightly different days.

In [ ]:
# Step 8.4: Error by forecast day for each multi-step experiment.
multi_names = [name for name, r in results.items() if r["horizon"] > 1]
by_day_rows = []
for name in multi_names:
    r = results[name]
    for split, label in [("val", "validation"), ("test", "test")]:
        for source, key in [("RNN", "model"), ("Baseline", "baseline_metrics")]:
            row = {"experiment": name, "period": label, "predictor": source}
            row.update({f"day {d + 1}": mae for d, mae in enumerate(r[split][key]["mae_by_day"])})
            by_day_rows.append(row)

if not by_day_rows:
    print("No multi-step experiments to compare.")
else:
    by_day = pd.DataFrame(by_day_rows).set_index(["experiment", "period", "predictor"])
    display(by_day.round(2))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, (split, label) in zip(axes, [("val", "Validation"), ("test", "Test")]):
        for name in multi_names:
            r = results[name]
            days = np.arange(1, r["horizon"] + 1)
            ax.plot(days, r[split]["model"]["mae_by_day"], color=styles[name]["color"],
                    marker=styles[name]["marker"], markersize=8, linewidth=2, label=f"RNN {name}")
            ax.plot(days, r[split]["baseline_metrics"]["mae_by_day"], color=BASELINE_COLOR,
                    linestyle="--", marker="o", markersize=8, linewidth=1.5,
                    label=f"Baseline ({name} windows)")
        ax.set_xticks(np.arange(1, max_horizon + 1))
        ax.set_xlabel("Forecast day")
        ax.set_title(f"{label} MAE by forecast day")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    axes[0].set_ylabel("MAE (USD)")
    plt.tight_layout()
    plt.show()

Five-day forecast paths during the test period. Each colored segment starts on the date of its first target and shows the five opening prices predicted from one window; only every fifth window is drawn so the paths do not overlap.

In [ ]:
# Step 8.4 (continued): Sample five-day forecast paths over the test period.
for name in multi_names:
    data = results[name]["test"]
    horizon = results[name]["horizon"]
    plt.figure(figsize=(11, 4.5))
    plt.plot(mu_test["Date"], mu_test["Open"], color=ACTUAL_COLOR, linewidth=1.5, label="Actual opening price")
    for i, start in enumerate(range(0, len(data["dates"]), horizon)):
        path_dates = mu_test["Date"].iloc[start:start + horizon]
        plt.plot(path_dates, data["predicted"][start, :len(path_dates)], color=styles[name]["color"],
                 marker=styles[name]["marker"], markersize=5, linewidth=2,
                 label=f"RNN {name} five-day forecast" if i == 0 else None)
    plt.title(f"Micron (MU) test period: {name} five-day forecasts")
    plt.xlabel("Date")
    plt.ylabel("Opening price (USD)")
    plt.legend()
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 9. Reload a saved checkpoint
Each experiment saved a checkpoint in `OUTPUT_DIR` with its weights, configuration, lookback, horizon, price transform, and fitted scaler parameters. The cell below rebuilds the one-day experiment with the lowest validation MAE (`best_name` from Section 8.1) from its file and verifies that it reproduces the stored predictions. Load only checkpoints you trust.

In [ ]:
# Step 9: Reconstruct the selected model and scaler from disk, then verify the round trip.
selected = results[best_name]
saved = torch.load(selected["checkpoint_path"], map_location="cpu", weights_only=True)
reloaded_model = MuRNN(**saved["model_config"])
reloaded_model.load_state_dict(saved["model_state_dict"])
reloaded_model = reloaded_model.to(device)
reloaded_model.eval()

reloaded_scaler = MinMaxScaler(feature_range=tuple(saved["scaler"]["feature_range"]))
for attr in ("scale_", "min_", "data_min_", "data_max_", "data_range_"):
    setattr(reloaded_scaler, attr, np.asarray(saved["scaler"][attr], dtype=np.float32))
reloaded_scaler.n_features_in_ = saved["scaler"]["n_features_in_"]
reloaded_scaler.n_samples_seen_ = saved["scaler"]["n_samples_seen_"]

X_check, _ = make_eval_windows(full_train_scaled, test_scaled, saved["lookback"], saved["horizon"])
scaled_check = predict_scaled(reloaded_model, X_check)
restored = reloaded_scaler.inverse_transform(scaled_check.reshape(-1, 1)).reshape(scaled_check.shape)
if saved["price_transform"] == "log":
    restored = np.exp(restored)
np.testing.assert_allclose(restored, selected["test"]["predicted"], rtol=1e-4)
print(f"Reloaded {best_name} from {selected['checkpoint_path'].name}; predictions match.")

## Interpretation checklist
- **Validation versus baseline.** Compare each `validation MAE / baseline` with 1. A model that tracks the price curve can still lose to the previous-open baseline.
- **Lookback.** Longer windows give more context but fewer training windows and longer gradient paths through time. Note whether validation error improves, worsens, or stays flat as the lookback grows.
- **Extrapolation.** If Step 3 reported scaled values above 1, check the validation panels in Section 8.3 for predictions that flatten below the actual prices.
- **Five-day targets.** Note how error grows from day 1 to day 5 for the model and for the baseline.
- **Sample size.** The test period is short, so small differences in test MAE are not reliable evidence of one configuration being better.